# Explorative Analyse: Arbeitsmarkt Regensburg (Jobcenter) — Teil 7: Finanzen (Zahlungsansprüche)

Erschließt die **einzige Quelle für tatsächliche Euro-Beträge** im gesamten Datensatz: Bearbeitet Priorität 4 aus dem Priorisierungsvorschlag in [`ArbeitslosenAnalyse.md`](../MD/ArbeitslosenAnalyse.md).

**Neue Datenquelle:** Tabellenblatt „3.6 Zahlungsansprüche“ (wie schon [`06_Erwerbstaetigkeit.ipynb`](06_Erwerbstaetigkeit.ipynb) direkt aus den Excel-Rohdaten gelesen, nicht aus `eckwerte_long.csv`). Anders als Blatt „3.3 Erwerbstätigkeit“ gibt es hier nur **einen** Datenblock pro Datei mit **3 Monaten Wartezeit** — die Zeitreihe ergibt sich daher wie bei Kapitel 6 für **Oktober 2024 – Juni 2026** (3 Monate früher als der Berichtsmonat der Datei).

In [1]:
import re
from pathlib import Path

import openpyxl
import pandas as pd
import plotly.graph_objects as go

RAW_DIR = Path("..") / "Data" / "RegensburgJCData"
PROCESSED_DIR = Path("..") / "Data" / "processed"

MONATE = {
    "Januar": 1, "Februar": 2, "März": 3, "April": 4, "Mai": 5, "Juni": 6,
    "Juli": 7, "August": 8, "September": 9, "Oktober": 10, "November": 11, "Dezember": 12,
}

# Feste Zeilen im Blatt "3.6 Zahlungsansprüche" (Spalte B = Betrag in Tsd. Euro)
ZEILEN = {
    38: "Gesamtregelleistung (Bürgergeld)",  # Teilsumme aus den vier folgenden Zeilen
    39: "Regelbedarf ELB",
    40: "Regelbedarf NEF",
    41: "Mehrbedarfe",
    42: "Kosten der Unterkunft",
    45: "Sozialversicherungsbeiträge",
    47: "weitere Zahlungsansprüche",
}


def parse_monat(text):
    monat_name, jahr = re.match(r"(\w+)\s+(\d{4})", text).groups()
    return pd.Period(f"{jahr}-{MONATE[monat_name]:02d}", freq="M")


def parse_zahlungsansprueche(path):
    wb = openpyxl.load_workbook(path, data_only=True)
    ws = wb["3.6 Zahlungsansprüche"]
    monat = parse_monat(ws["A5"].value)
    zeile = {
        "period": monat,
        "Insgesamt (Tsd. Euro)": ws.cell(row=36, column=2).value,
        "Ø je BG insgesamt (Euro)": ws.cell(row=36, column=5).value,
        "Bestand BG": ws.cell(row=36, column=7).value,
    }
    for r, label in ZEILEN.items():
        zeile[label] = ws.cell(row=r, column=2).value
    wb.close()
    return zeile


finanzen = pd.DataFrame(
    [parse_zahlungsansprueche(p) for p in sorted(RAW_DIR.glob("*.xlsx"))]
).set_index("period").sort_index()

kennzahlen = pd.read_csv(PROCESSED_DIR / "kennzahlen_sgb2.csv", index_col="period", parse_dates=["period"])
kennzahlen = kennzahlen.sort_index()
kennzahlen.index = kennzahlen.index.to_period("M")

finanzen.tail()

ABBILDUNGEN_DIR = Path("..") / "Docs" / "Abbildungen"


def export_abbildung(fig, dateiname, width, height, **layout):
    """Speichert eine Kopie der Grafik im weissen Layout der Projektdokumentation
    (Docs/Abbildungen, doppelte Aufloesung); die Anzeige im Notebook bleibt unveraendert.
    Optionale `layout`-Angaben (z. B. kuerzerer Titel) gelten nur fuer die exportierte Datei."""
    go.Figure(fig).update_layout(template="plotly_white", **layout).write_image(
        ABBILDUNGEN_DIR / dateiname, scale=2, width=width, height=height
    )

C:\Users\funke\AppData\Roaming\Python\Python314\site-packages\openpyxl\reader\drawings.py:33: UserWarning: DrawingML support is incomplete and limited to charts and images only. Shapes and drawings will be lost.
  warn("DrawingML support is incomplete and limited to charts and images only. Shapes and drawings will be lost.")
C:\Users\funke\AppData\Roaming\Python\Python314\site-packages\openpyxl\reader\drawings.py:67: UserWarning: wmf image format is not supported so the image is being dropped
  warn(msg)


## 1. Datenüberblick und Cross-Check gegen die eigene BG-Zeitreihe

Das Blatt weist neben den Euro-Beträgen auch den **Bestand an Bedarfsgemeinschaften (BG)** aus, für den ein Zahlungsanspruch besteht. Da wir die Zahl der Bedarfsgemeinschaften bereits unabhängig davon aus dem Blatt „1.1 Eckwerte“ kennen ([`kennzahlen_sgb2.csv`](../Data/processed/kennzahlen_sgb2.csv)), lässt sich das als Plausibilitätscheck nutzen — für die 18 Monate, in denen sich beide Zeitreihen überlappen (Januar 2025 – Juni 2026).

In [2]:
print(f"Anzahl Monate: {len(finanzen)}")
print(f"Zeitraum: {finanzen.index.min()} bis {finanzen.index.max()}")
print(f"Duplikate: {finanzen.index.duplicated().sum()}")

leaf_spalten = ["Regelbedarf ELB", "Regelbedarf NEF", "Mehrbedarfe", "Kosten der Unterkunft", "Sozialversicherungsbeiträge", "weitere Zahlungsansprüche"]
summen_check = finanzen[leaf_spalten].sum(axis=1) - finanzen["Insgesamt (Tsd. Euro)"]
print(f"Summen-Check (6 Einzelposten vs. Insgesamt): min={summen_check.min():.3f}, max={summen_check.max():.3f}")

overlap = finanzen.index.intersection(kennzahlen.index)
cross_check = pd.DataFrame({
    "Bestand BG (Zahlungsansprüche-Blatt)": finanzen.loc[overlap, "Bestand BG"],
    "Bedarfsgemeinschaften (eigene Zeitreihe)": kennzahlen.loc[overlap, "Bedarfsgemeinschaften"],
})
cross_check["Differenz"] = cross_check["Bestand BG (Zahlungsansprüche-Blatt)"] - cross_check["Bedarfsgemeinschaften (eigene Zeitreihe)"]
cross_check.round(1)

Anzahl Monate: 21
Zeitraum: 2024-10 bis 2026-06
Duplikate: 0
Summen-Check (6 Einzelposten vs. Insgesamt): min=-0.000, max=0.000


,Bestand BG (Zahlungsansprüche-Blatt),Bedarfsgemeinschaften (eigene Zeitreihe),Differenz
period,,,
2025-01,2791,2797.3,-6.3
2025-02,2775,2755.8,19.2
2025-03,2749,2781.4,-32.4
2025-04,2720,2767.0,-47.0
2025-05,2698,2707.6,-9.6
2025-06,2686,2696.6,-10.6
2025-07,2666,2616.9,49.1
2025-08,2683,2688.1,-5.1
2025-09,2667,2669.3,-2.3


Alle 21 Dateien liefern 21 lückenlose, eindeutige Monate (Oktober 2024 – Juni 2026, keine Duplikate). Der Summen-Check bestätigt: Die sechs Einzelposten summieren sich in jedem Monat exakt zu „Insgesamt“ (Differenz < 0,001 Tsd. Euro — reine Rundung). Der Cross-Check des „Bestand BG“ aus diesem Blatt gegen unsere eigene Bedarfsgemeinschaften-Zeitreihe zeigt eine kleine, aber nicht systematische Abweichung (Differenz im Schnitt −12,5, Spanne −47 bis +49 Bedarfsgemeinschaften) — plausibel durch leicht unterschiedliche Stichtage bzw. nachträgliche Revisionen der beiden Blätter, kein Hinweis auf einen Parsing-Fehler.

## 2. Gesamtvolumen der Zahlungsansprüche im Zeitverlauf

In [3]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=finanzen.index.to_timestamp(), y=finanzen["Insgesamt (Tsd. Euro)"], name="Insgesamt (Tsd. Euro)",
    mode="lines+markers", line=dict(color="#2563eb"), marker=dict(size=6), yaxis="y1",
))
fig.add_trace(go.Scatter(
    x=finanzen.index.to_timestamp(), y=finanzen["Ø je BG insgesamt (Euro)"], name="Ø je BG (Euro)",
    mode="lines+markers", line=dict(color="#f59e0b", dash="dot"), marker=dict(size=6), yaxis="y2",
))
fig.update_layout(
    title="Zahlungsansprüche für Bedarfsgemeinschaften im Zeitverlauf (SGB II)",
    yaxis=dict(title="Insgesamt (Tsd. Euro)"),
    yaxis2=dict(title="Ø je BG (Euro)", overlaying="y", side="right"),
    xaxis_title="Monat",
    hovermode="x unified",
    legend=dict(orientation="h", y=1.15),
    height=460,
)
export_abbildung(fig, "abb20_zahlungsanspruch_volumen.png", 1000, 480)
fig

Das Gesamtvolumen der Zahlungsansprüche ist über den Zeitraum bemerkenswert stabil: 3.451,5 Tsd. Euro (Oktober 2024) zu 3.443,1 Tsd. Euro (Juni 2026) — praktisch unverändert, mit einer Schwankungsbreite von rund 3.303 bis 3.540 Tsd. Euro dazwischen. Der **Durchschnitt je Bedarfsgemeinschaft** ist dagegen spürbar gestiegen: von 1.251 € auf 1.299 € (Ø 1.272 €, Spanne 1.242–1.318 €) — da die Zahl der Bedarfsgemeinschaften über den Zeitraum sinkt (siehe [`02_Analyse_SGBII.ipynb`](02_Analyse_SGBII.ipynb)), das Gesamtvolumen aber kaum, steigt der Anspruch pro verbleibender Bedarfsgemeinschaft.

## 3. Struktur der Zahlungsansprüche

Sechs sich nicht überschneidende Einzelposten, deren Summe exakt „Insgesamt“ ergibt (siehe Summen-Check in Kapitel 1). „Gesamtregelleistung (Bürgergeld)“ ist dagegen eine **Teilsumme** aus Regelbedarf ELB + Regelbedarf NEF + Mehrbedarfe + Kosten der Unterkunft und wird hier nicht separat gestapelt, um Doppelzählung zu vermeiden.

In [4]:
leaf_farben = {
    "Regelbedarf ELB": "#2563eb", "Regelbedarf NEF": "#7c3aed", "Mehrbedarfe": "#db2777",
    "Kosten der Unterkunft": "#f59e0b", "Sozialversicherungsbeiträge": "#16a34a", "weitere Zahlungsansprüche": "#6b7280",
}

fig = go.Figure()
for spalte in leaf_spalten:
    fig.add_trace(go.Scatter(
        x=finanzen.index.to_timestamp(), y=finanzen[spalte], name=spalte,
        mode="lines", stackgroup="finanzen", line=dict(color=leaf_farben[spalte], width=0.5),
    ))
fig.update_layout(
    title="Struktur der Zahlungsansprüche (Tsd. Euro)",
    yaxis_title="Tsd. Euro",
    xaxis_title="Monat",
    hovermode="x unified",
    legend=dict(orientation="h", y=1.15),
    height=460,
)
export_abbildung(fig, "abb21_zahlungsanspruch_struktur.png", 1000, 460)
fig

Die Struktur ist über den Zeitraum größtenteils stabil, mit zwei erwähnenswerten Verschiebungen: **Regelbedarf ELB** bleibt mit Abstand der größte Posten, sinkt aber leicht von 41,6 % auf 39,4 % Anteil am Gesamtvolumen. **Kosten der Unterkunft** ist mit rund 36 % (35,8 % → 36,5 %) nahezu konstant der zweitgrößte Posten. Am auffälligsten: **Sozialversicherungsbeiträge** steigen deutlich von 15,6 % auf 18,0 % Anteil (Höchstwert 18,6 %) — während **Regelbedarf NEF** (v. a. Kinder) von 3,8 % auf 2,8 % sinkt, passend zum insgesamt rückläufigen Anteil nicht erwerbsfähiger Leistungsberechtigter.

## 4. Export der aufbereiteten Daten

Grundlage für den Dashboard-Tab „💶 Finanzen“.

In [5]:
finanzen.to_csv(PROCESSED_DIR / "finanzen_sgb2.csv")
finanzen.tail()

,Insgesamt (Tsd. Euro),Ø je BG insgesamt (Euro),Bestand BG,Gesamtregelleistung (Bürgergeld),Regelbedarf ELB,Regelbedarf NEF,Mehrbedarfe,Kosten der Unterkunft,Sozialversicherungsbeiträge,weitere Zahlungsansprüche
period,,,,,,,,,,
2026-02,3403.95866,1282.093657,2651,2752.16585,1355.77811,96.23157,77.50061,1222.65556,629.28562,22.50719
2026-03,3472.99066,1288.201283,2694,2816.25470,1365.30094,98.60636,76.84839,1275.49901,635.16762,21.56834
2026-04,3429.78393,1279.292775,2680,2774.95079,1364.52525,97.19524,77.05136,1236.17894,629.61744,25.21570
2026-05,3452.58599,1284.922214,2683,2795.79998,1366.74046,96.02393,77.77439,1255.26120,627.87560,28.91041
2026-06,3443.09524,1298.791113,2646,2788.28781,1357.34060,96.10165,78.01016,1256.83540,618.04507,36.76236


## 5. Zusammenfassung

- **Gesamtvolumen:** mit 3.451,5 → 3.443,1 Tsd. Euro über den Zeitraum praktisch unverändert (Schwankungsbreite ca. 3.303–3.540 Tsd. Euro).
- **Ø je Bedarfsgemeinschaft:** deutlich gestiegen (1.251 € → 1.299 €, Ø 1.272 €) — weil die Zahl der Bedarfsgemeinschaften sinkt, das Gesamtvolumen aber kaum.
- **Struktur:** Regelbedarf ELB (größter Posten, leicht rückläufig 41,6 % → 39,4 %) und Kosten der Unterkunft (~36 %, stabil) dominieren gemeinsam rund drei Viertel der Zahlungsansprüche. Sozialversicherungsbeiträge steigen spürbar (15,6 % → 18,0 %), Regelbedarf NEF sinkt (3,8 % → 2,8 %).
- **Datenqualität:** Cross-Check des Bedarfsgemeinschaften-Bestands gegen die eigene Zeitreihe zeigt nur kleine, nicht systematische Abweichungen — die Daten sind konsistent.
- **Datenbesonderheit:** Zeitreihe umfasst Oktober 2024 – Juni 2026 (3 Monate vor dem jeweiligen Berichtsmonat), analog zu [`06_Erwerbstaetigkeit.ipynb`](06_Erwerbstaetigkeit.ipynb).

Insgesamt zeigt sich: Trotz sinkender Zahl an Bedarfsgemeinschaften bleibt das insgesamt ausgezahlte Leistungsvolumen stabil — der Anspruch pro verbleibender Bedarfsgemeinschaft steigt, mit einer spürbaren Verschiebung hin zu höheren Sozialversicherungsbeiträgen.

Weiter geht es mit der Umsetzung im [Streamlit-Dashboard](../Dashboard/app.py) (Tab „💶 Finanzen“) sowie optional mit `08_Foerderung.ipynb` (aktive Arbeitsmarktpolitik, ⚠️ 3 Monate Wartezeit).